# Stroke Detection from Brain CT — **DICOM pipeline**

End-to-end deep learning notebook that reads raw **DICOM** (`.dcm`) files from the Brain Stroke CT Dataset and classifies each slice as **Bleeding**, **Ischemia**, or **Normal**.

### Why DICOM instead of PNG?
The exported PNGs compress the original 12-bit CT intensities into 8 bits and bake-in a single window setting, which destroys diagnostically useful contrast. Working directly from DICOM lets us:

1. Recover **Hounsfield Units (HU)** using the `RescaleSlope` / `RescaleIntercept` DICOM tags.
2. Apply a **clinical window** (e.g., brain window: center = 40 HU, width = 80 HU) that matches what radiologists use to spot strokes.
3. Keep full control over resolution, normalization, and channel layout.

### Notebook structure
1. Imports
2. DICOM Loading & EDA
3. Preprocessing (HU rescale → windowing → normalization → split)
4. Model Building
5. Training
6. Evaluation
7. External test set


## 1. Imports

`pydicom` reads `.dcm` files into NumPy arrays. Everything else is the same deep-learning stack (TensorFlow/Keras, scikit-learn, matplotlib, seaborn).

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pydicom
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, callbacks
from tensorflow.keras.utils import to_categorical

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("pydicom  :", pydicom.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)


## 2. DICOM Loading & EDA

We scan each class folder under `Brain_Stroke_CT_Dataset/<class>/DICOM` and record every `.dcm` file path with its label. No pixels are loaded here — only paths — so this stays cheap.

In [ ]:
DATA_DIR = "Brain_Stroke_CT_Dataset"
CLASSES = ["Bleeding", "Ischemia", "Normal"]

records = []
for cls in CLASSES:
    dcm_dir = os.path.join(DATA_DIR, cls, "DICOM")
    for p in sorted(glob.glob(os.path.join(dcm_dir, "*.dcm"))):
        records.append({"path": p, "label": cls})

df = pd.DataFrame(records)
print("Total DICOM files:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts())
df.head()


### Inspect a single DICOM header

A DICOM file carries a rich header. The tags we care about for preprocessing are:

| Tag | Purpose |
|---|---|
| `Rows`, `Columns` | Raw image resolution |
| `BitsStored` | Usually 12 for CT, stored as int16 |
| `RescaleSlope`, `RescaleIntercept` | Convert raw pixels → **Hounsfield Units (HU)** via `HU = pixel * slope + intercept` |
| `WindowCenter`, `WindowWidth` | The window the scanner metadata suggests (we override with a clinical brain window) |
| `PhotometricInterpretation` | `MONOCHROME2` means higher value = brighter, as expected |

In [ ]:
sample_path = df.iloc[0]["path"]
ds = pydicom.dcmread(sample_path)

for tag in [
    "Modality",
    "Rows",
    "Columns",
    "BitsStored",
    "PixelRepresentation",
    "RescaleSlope",
    "RescaleIntercept",
    "WindowCenter",
    "WindowWidth",
    "PhotometricInterpretation",
]:
    print(f"{tag:>26}: {getattr(ds, tag, 'N/A')}")

arr = ds.pixel_array
print("\nRaw pixel array:", arr.shape, arr.dtype, "min=", arr.min(), "max=", arr.max())


In [ ]:
# Class distribution bar plot
counts = df["label"].value_counts().reindex(CLASSES)

plt.figure(figsize=(6, 4))
sns.barplot(x=counts.index, y=counts.values, palette="viridis")
plt.title("Class Distribution (DICOM)")
plt.ylabel("Number of slices")
for i, v in enumerate(counts.values):
    plt.text(i, v + 30, str(v), ha="center", fontweight="bold")
plt.show()

print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}x")


### Visualize the effect of HU windowing

A brain CT scan contains tissues spanning roughly **-1000 HU (air) to +1000 HU (bone)**. Stroke findings live in a very narrow band near soft tissue (0–80 HU). If we just rescale the whole range to 8-bit we drown the signal in bone and air.

The three standard windows used here:

- **Brain window** (C=40, W=80) — best for ischemia (subtle grey/white matter contrast).
- **Stroke / subdural window** (C=40, W=380) — wider, highlights acute bleeds.
- **Bone window** (C=600, W=2800) — shows skull fractures.

The cell below shows the same slice under each window + the raw display.

In [ ]:
# Convert a sample slice to HU and display under several windows
ds = pydicom.dcmread(sample_path)
slope = float(getattr(ds, "RescaleSlope", 1.0))
intercept = float(getattr(ds, "RescaleIntercept", 0.0))
hu = ds.pixel_array.astype(np.float32) * slope + intercept
print(f"HU range for sample slice: [{hu.min():.0f}, {hu.max():.0f}]")

windows = {
    "Raw (no window)": (hu.min() + (hu.max() - hu.min()) / 2, hu.max() - hu.min()),
    "Brain (C=40, W=80)": (40, 80),
    "Stroke/Subdural (C=40, W=380)": (40, 380),
    "Bone (C=600, W=2800)": (600, 2800),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, (c, w)) in zip(axes, windows.items()):
    lo, hi = c - w / 2, c + w / 2
    img = np.clip((hu - lo) / (hi - lo), 0.0, 1.0)
    ax.imshow(img, cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# One DICOM sample per class under the brain window, as a sanity check
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row, cls in enumerate(CLASSES):
    samples = df[df["label"] == cls].sample(4, random_state=SEED)["path"].tolist()
    for col, p in enumerate(samples):
        d = pydicom.dcmread(p)
        s = float(getattr(d, "RescaleSlope", 1.0))
        b = float(getattr(d, "RescaleIntercept", 0.0))
        hu = d.pixel_array.astype(np.float32) * s + b
        lo, hi = 40 - 40, 40 + 40  # brain window
        img = np.clip((hu - lo) / (hi - lo), 0.0, 1.0)
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].set_title(f"{cls} ({d.Rows}x{d.Columns})", fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


## 3. Preprocessing

For every DICOM we will:

1. Read the pixel array with `pydicom.dcmread(...).pixel_array`.
2. Apply the DICOM rescale: `HU = pixels * RescaleSlope + RescaleIntercept`.
3. Stack **three clinical windows** (brain / stroke / bone) as 3 channels. This gives the CNN a "multi-view" input similar to what a radiologist sees when toggling windows, and also means we can use ImageNet-style 3-channel models later.
4. Resize to `224 × 224`.
5. Build a stratified 70 / 15 / 15 train / val / test split.


In [ ]:
IMG_SIZE = 224

# (center, width) for each channel — brain, stroke/subdural, bone
WINDOWS = [(40, 80), (40, 380), (600, 2800)]

X = np.zeros((len(df), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
y_raw = df["label"].values

failed = []
for i, p in enumerate(df["path"].values):
    try:
        d = pydicom.dcmread(p)
        arr = d.pixel_array.astype(np.float32)
        slope = float(getattr(d, "RescaleSlope", 1.0))
        intercept = float(getattr(d, "RescaleIntercept", 0.0))
        hu = arr * slope + intercept

        channels = []
        for c, w in WINDOWS:
            lo, hi = c - w / 2, c + w / 2
            ch = np.clip((hu - lo) / (hi - lo), 0.0, 1.0)
            ch = np.asarray(
                Image.fromarray((ch * 255).astype(np.uint8)).resize(
                    (IMG_SIZE, IMG_SIZE), Image.BILINEAR
                ),
                dtype=np.float32,
            ) / 255.0
            channels.append(ch)
        X[i] = np.stack(channels, axis=-1)
    except Exception as e:
        failed.append((p, str(e)))
        # leave as zeros; will drop below

    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1}/{len(df)}")

print("\nFailed DICOMs:", len(failed))
print("X shape:", X.shape, "dtype:", X.dtype)
print("Pixel range:", X.min(), "to", X.max())


In [ ]:
# Label encoding + one-hot
label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(y_raw)
NUM_CLASSES = len(label_encoder.classes_)
y_onehot = to_categorical(y_int, num_classes=NUM_CLASSES)

print("Classes:", list(label_encoder.classes_))
print("One-hot shape:", y_onehot.shape)

# Stratified 70 / 15 / 15 split
X_train, X_temp, y_train, y_temp, y_train_int, y_temp_int = train_test_split(
    X, y_onehot, y_int, test_size=0.30, stratify=y_int, random_state=SEED
)
X_val, X_test, y_val, y_test, y_val_int, y_test_int = train_test_split(
    X_temp, y_temp, y_temp_int, test_size=0.50, stratify=y_temp_int, random_state=SEED
)
print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_test.shape[0]}")

# Class weights
cw = compute_class_weight(class_weight="balanced", classes=np.unique(y_train_int), y=y_train_int)
class_weights = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", {label_encoder.classes_[i]: round(w, 3) for i, w in class_weights.items()})


## 4. Model Building

A compact CNN that takes the **3-channel windowed CT** as input. Each Conv block is `Conv → BatchNorm → MaxPool`. BatchNorm stabilizes training on limited data, `GlobalAveragePooling2D` avoids a huge dense layer, and two Dropout layers regularize the classifier head.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="ct_windows")
x = data_augmentation(inputs)

# --- Stem ---
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)

# =========================
# Block 1
# =========================
shortcut = x

x = layers.SeparableConv2D(32, 3, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)

x = layers.SeparableConv2D(32, 3, padding="same")(x)
x = layers.BatchNormalization()(x)

x = layers.Add()([x, shortcut])
x = layers.Activation("relu")(x)

# Spatial Attention
avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(x)
max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(x)
attn = layers.Concatenate()([avg_pool, max_pool])
attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(attn)
x = layers.Multiply()([x, attn])


# =========================
# Block 2 (Downsample)
# =========================
shortcut = layers.Conv2D(64, 1, strides=2, padding="same")(x)
shortcut = layers.BatchNormalization()(shortcut)

x = layers.SeparableConv2D(64, 3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)

x = layers.SeparableConv2D(64, 3, padding="same")(x)
x = layers.BatchNormalization()(x)

x = layers.Add()([x, shortcut])
x = layers.Activation("relu")(x)

# Spatial Attention
avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(x)
max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(x)
attn = layers.Concatenate()([avg_pool, max_pool])
attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(attn)
x = layers.Multiply()([x, attn])


# =========================
# Block 3 (Downsample)
# =========================
shortcut = layers.Conv2D(128, 1, strides=2, padding="same")(x)
shortcut = layers.BatchNormalization()(shortcut)

x = layers.SeparableConv2D(128, 3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)

x = layers.SeparableConv2D(128, 3, padding="same")(x)
x = layers.BatchNormalization()(x)

x = layers.Add()([x, shortcut])
x = layers.Activation("relu")(x)

# Spatial Attention
avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(x)
max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(x)
attn = layers.Concatenate()([avg_pool, max_pool])
attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(attn)
x = layers.Multiply()([x, attn])


# =========================
# Block 4 (Downsample)
# =========================
shortcut = layers.Conv2D(256, 1, strides=2, padding="same")(x)
shortcut = layers.BatchNormalization()(shortcut)

x = layers.SeparableConv2D(256, 3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)

x = layers.SeparableConv2D(256, 3, padding="same")(x)
x = layers.BatchNormalization()(x)

x = layers.Add()([x, shortcut])
x = layers.Activation("relu")(x)

# Spatial Attention
avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(x)
max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(x)
attn = layers.Concatenate()([avg_pool, max_pool])
attn = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(attn)
x = layers.Multiply()([x, attn])


# =========================
# Head
# =========================
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs, name="StrokeCNN_Intermediate")
model.summary()

## 5. Training

- **Loss**: categorical cross-entropy (multi-class one-hot).
- **Optimizer**: Adam @ `1e-3`.
- **Callbacks**: `EarlyStopping` on `val_loss` with weight restoration, `ModelCheckpoint` to save the best model, and `ReduceLROnPlateau` to drop the LR if validation loss stalls.
- **Class weights** compensate for the imbalance we saw above.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

os.makedirs("checkpoints", exist_ok=True)
cb_list = [
    callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True, verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath="checkpoints/stroke_cnn_dicom_best.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

EPOCHS = 20
BATCH_SIZE = 32

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=cb_list,
    verbose=1,
)


## 6. Evaluation

We look at four things: learning curves, overall test accuracy, per-class precision/recall/F1, and the confusion matrix (both counts and row-normalized).

In [ ]:
# Training curves
hist = history.history
epochs_range = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_range, hist["accuracy"], label="train")
axes[0].plot(epochs_range, hist["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_range, hist["loss"], label="train")
axes[1].plot(epochs_range, hist["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Test-set metrics
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = y_test_int
target_names = list(label_encoder.classes_)

print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))


In [ ]:
# Confusion matrices: raw and row-normalized
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(np.float32) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(cm, display_labels=target_names).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title("Confusion Matrix (counts)")

sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=target_names, yticklabels=target_names,
    ax=axes[1], cbar=False,
)
axes[1].set_title("Confusion Matrix (row-normalized)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
plt.tight_layout()
plt.show()


In [ ]:
# Inspect misclassified examples (show the brain-window channel only)
mis_idx = np.where(y_pred != y_true)[0]
print(f"Misclassified: {len(mis_idx)} / {len(y_true)}")

n_show = min(8, len(mis_idx))
if n_show > 0:
    sel = np.random.RandomState(SEED).choice(mis_idx, n_show, replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, i in zip(axes.ravel(), sel):
        ax.imshow(X_test[i, ..., 0], cmap="gray")  # channel 0 = brain window
        ax.set_title(
            f"T: {target_names[y_true[i]]}\n"
            f"P: {target_names[y_pred[i]]} ({y_prob[i, y_pred[i]]:.2f})",
            fontsize=9,
        )
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
model.save("stroke_cnn_dicom_model.h5")

## 7. External test set

`Brain_Stroke_CT_Dataset/External_Test` holds held-out DICOMs with a `labels.csv`. Running the trained model here gives an honest generalization estimate, since these slices were never seen during training or model selection.

The external `labels.csv` uses **binary** labels (`Stroke` = 0 / 1), so we collapse our 3-class predictions into *Stroke* (Bleeding ∪ Ischemia) vs *Normal* for a fair comparison.

In [ ]:
ext_dir = os.path.join(DATA_DIR, "External_Test")
ext_labels = pd.read_csv(os.path.join(ext_dir, "labels.csv"))
print("External set size:", len(ext_labels))
print(ext_labels["Stroke"].value_counts())

# Build paths. image_id corresponds to "<id>.dcm" under External_Test/DICOM/
ext_labels["path"] = ext_labels["image_id"].astype(str).apply(
    lambda i: os.path.join(ext_dir, "DICOM", f"{i}.dcm")
)
ext_labels = ext_labels[ext_labels["path"].apply(os.path.exists)].reset_index(drop=True)
print("Resolvable files:", len(ext_labels))

X_ext = np.zeros((len(ext_labels), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
for i, p in enumerate(ext_labels["path"].values):
    d = pydicom.dcmread(p)
    arr = d.pixel_array.astype(np.float32)
    slope = float(getattr(d, "RescaleSlope", 1.0))
    intercept = float(getattr(d, "RescaleIntercept", 0.0))
    hu = arr * slope + intercept
    chs = []
    for c, w in WINDOWS:
        lo, hi = c - w / 2, c + w / 2
        ch = np.clip((hu - lo) / (hi - lo), 0.0, 1.0)
        ch = np.asarray(
            Image.fromarray((ch * 255).astype(np.uint8)).resize(
                (IMG_SIZE, IMG_SIZE), Image.BILINEAR
            ),
            dtype=np.float32,
        ) / 255.0
        chs.append(ch)
    X_ext[i] = np.stack(chs, axis=-1)

y_ext_true = ext_labels["Stroke"].values  # 0 = Normal, 1 = Stroke
print("X_ext:", X_ext.shape)


In [ ]:
# Collapse 3-class probs -> binary stroke probability (P(Bleeding) + P(Ischemia))
y_ext_prob3 = model.predict(X_ext, batch_size=BATCH_SIZE, verbose=0)
normal_idx = list(label_encoder.classes_).index("Normal")
p_stroke = 1.0 - y_ext_prob3[:, normal_idx]
y_ext_pred = (p_stroke >= 0.5).astype(int)

print("External binary accuracy:",
      float((y_ext_pred == y_ext_true).mean()))
print("\nClassification report (external, binary):\n")
print(classification_report(y_ext_true, y_ext_pred, target_names=["Normal", "Stroke"], digits=4))

cm_ext = confusion_matrix(y_ext_true, y_ext_pred)
ConfusionMatrixDisplay(cm_ext, display_labels=["Normal", "Stroke"]).plot(cmap="Blues", colorbar=False)
plt.title("External Test — Binary Confusion Matrix")
plt.show()


## 8. Skull stripping / intracranial-ROI cropping

A lot of each slice is **background air** (≈ -1000 HU) and **skull bone** (>300 HU). Both are irrelevant for stroke detection and introduce nuisance variance (different FOVs, patient positioning, black borders baked in from the scanner). We build a quick **brain mask** and zero-out everything outside it:

1. Threshold the brain-window channel to get rough tissue pixels.
2. Fill holes (ventricles, dark regions inside the skull).
3. Keep only the **largest connected component** — that is the intracranial region.
4. Multiply the 3-channel tensor by the mask so the model sees brain-only inputs.

This is a lightweight, classical preprocessing step (no separate model needed). A proper skull-strip would use tools like BET or HD-BET on 3D volumes, but on 2D slices this mask is usually sufficient to remove external variance.


In [ ]:
from scipy import ndimage as ndi

# Build a brain mask from the brain-window channel (channel 0 of X).
# Pixels inside the skull have non-trivial intensity in the brain window;
# background air is ~0 after clipping.
MASK_THRESH = 0.02
STRUCT = np.ones((3, 3), dtype=bool)  # 8-connectivity

# Quick visual sanity check on one slice
sample = X_test[0]
brain_ch = sample[..., 0]
m = brain_ch > MASK_THRESH
m = ndi.binary_fill_holes(m)
lab, n = ndi.label(m, structure=STRUCT)
if n > 0:
    sizes = ndi.sum(m, lab, index=np.arange(1, n + 1))
    keep = np.argmax(sizes) + 1
    m = lab == keep
m = ndi.binary_fill_holes(m)
m_eroded = ndi.binary_erosion(m, iterations=2)  # pull away from skull rim

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(sample[..., 0], cmap="gray"); axes[0].set_title("Brain window"); axes[0].axis("off")
axes[1].imshow(m, cmap="gray"); axes[1].set_title("Largest CC mask"); axes[1].axis("off")
axes[2].imshow(m_eroded, cmap="gray"); axes[2].set_title("Eroded mask"); axes[2].axis("off")
axes[3].imshow(sample[..., 0] * m_eroded, cmap="gray"); axes[3].set_title("Masked brain"); axes[3].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# Apply the same mask procedure to every tensor (train / val / test / external)
# so downstream models see background-suppressed inputs.
X_train_m = np.empty_like(X_train)
X_val_m   = np.empty_like(X_val)
X_test_m  = np.empty_like(X_test)
X_ext_m   = np.empty_like(X_ext)

for name, src, dst in [
    ("train", X_train, X_train_m),
    ("val",   X_val,   X_val_m),
    ("test",  X_test,  X_test_m),
    ("ext",   X_ext,   X_ext_m),
]:
    for i in range(len(src)):
        mm = src[i, ..., 0] > MASK_THRESH
        mm = ndi.binary_fill_holes(mm)
        lab, n = ndi.label(mm, structure=STRUCT)
        if n > 0:
            sizes = ndi.sum(mm, lab, index=np.arange(1, n + 1))
            mm = lab == (np.argmax(sizes) + 1)
            mm = ndi.binary_fill_holes(mm)
            mm = ndi.binary_erosion(mm, iterations=2)
        dst[i] = src[i] * mm.astype(np.float32)[..., None]
    print(f"{name:>5}: masked {len(src)} slices")

print("Shapes:", X_train_m.shape, X_val_m.shape, X_test_m.shape, X_ext_m.shape)


In [ ]:
# Visualize a few masked vs original slices per class
plt.figure(figsize=(14, 6))
for col in range(6):
    idx = np.random.RandomState(SEED + col).randint(len(X_train))
    plt.subplot(2, 6, col + 1)
    plt.imshow(X_train[idx, ..., 0], cmap="gray"); plt.axis("off")
    if col == 0: plt.ylabel("Original")
    plt.title(target_names[y_train_int[idx]], fontsize=9)

    plt.subplot(2, 6, col + 7)
    plt.imshow(X_train_m[idx, ..., 0], cmap="gray"); plt.axis("off")
    if col == 0: plt.ylabel("Masked")
plt.tight_layout(); plt.show()


## 9. Transfer learning — EfficientNetB0 / ResNet50

Our 3-channel windowed CT input (brain / stroke / bone) is shape-compatible with ImageNet pretrained models. A pretrained backbone gives strong low-level feature extractors (edges, textures) for free, which usually beats a small custom CNN on limited medical data.

Two-phase training:

1. **Head training** — freeze the backbone, train only the new classifier head at `1e-3`.
2. **Fine-tuning** — unfreeze the top ~30% of the backbone and continue at a much smaller LR (`1e-5`) so we don't destroy the pretrained features.

Switch `BACKBONE` between `"efficientnet"` and `"resnet50"` to compare. Both expect 3-channel inputs in `[0, 255]`, so we rescale our `[0, 1]` tensors and apply the model's own `preprocess_input`.


In [ ]:
# Fix: macOS Python can't verify SSL certs by default, so Keras fails to download
# the ImageNet weights. Point urllib at certifi's CA bundle before the backbone loads.
import os, ssl, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
print("CA bundle:", certifi.where())


In [ ]:
from tensorflow.keras.applications import EfficientNetB0, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as res_preprocess

BACKBONE = "efficientnet"  # or "resnet50"

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="ct_windows")
x = data_augmentation(inputs)
x = layers.Rescaling(255.0)(x)  # our tensors live in [0,1]; ImageNet models expect [0,255]

if BACKBONE == "efficientnet":
    x = layers.Lambda(eff_preprocess, name="eff_preprocess")(x)
    base = EfficientNetB0(include_top=False, weights="imagenet", input_tensor=x)
else:
    x = layers.Lambda(res_preprocess, name="res_preprocess")(x)
    base = ResNet50(include_top=False, weights="imagenet", input_tensor=x)

base.trainable = False  # phase 1: freeze backbone

h = layers.GlobalAveragePooling2D()(base.output)
h = layers.Dropout(0.3)(h)
h = layers.Dense(128, activation="relu")(h)
h = layers.Dropout(0.3)(h)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(h)

tl_model = models.Model(inputs, outputs, name=f"{BACKBONE}_stroke")
tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print(f"Backbone: {BACKBONE}  |  trainable params (head only): {sum([np.prod(v.shape) for v in tl_model.trainable_weights]):,}")
tl_model.summary()


In [ ]:
# Phase 1: train the head on the skull-stripped tensors
os.makedirs("checkpoints", exist_ok=True)
tl_cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        filepath=f"checkpoints/{BACKBONE}_head.keras",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

history_head = tl_model.fit(
    X_train_m, y_train,
    validation_data=(X_val_m, y_val),
    epochs=15,
    batch_size=32,
    class_weight=class_weights,
    callbacks=tl_cbs,
    verbose=1,
)


In [ ]:
# Phase 2: fine-tune the top ~30% of the backbone at a small LR
base.trainable = True
n_layers = len(base.layers)
freeze_until = int(n_layers * 0.7)
for lyr in base.layers[:freeze_until]:
    lyr.trainable = False
print(f"Unfrozen layers: {n_layers - freeze_until}/{n_layers}")

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

tl_cbs_ft = [
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        filepath=f"checkpoints/{BACKBONE}_finetuned.keras",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]

history_ft = tl_model.fit(
    X_train_m, y_train,
    validation_data=(X_val_m, y_val),
    epochs=10,
    batch_size=16,
    class_weight=class_weights,
    callbacks=tl_cbs_ft,
    verbose=1,
)


In [ ]:
# Evaluate the transfer-learning model on the masked internal test set
tl_loss, tl_acc = tl_model.evaluate(X_test_m, y_test, verbose=0)
print(f"[{BACKBONE}] Test loss: {tl_loss:.4f}  |  Test accuracy: {tl_acc:.4f}")

y_prob_tl = tl_model.predict(X_test_m, batch_size=32, verbose=0)
y_pred_tl = np.argmax(y_prob_tl, axis=1)

print("\nClassification report:\n")
print(classification_report(y_test_int, y_pred_tl, target_names=target_names, digits=4))

cm_tl = confusion_matrix(y_test_int, y_pred_tl)
cm_tl_norm = cm_tl.astype(np.float32) / cm_tl.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(cm_tl, display_labels=target_names).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"{BACKBONE} — Confusion (counts)")
sns.heatmap(cm_tl_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names, ax=axes[1], cbar=False)
axes[1].set_title(f"{BACKBONE} — Confusion (normalized)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
plt.tight_layout(); plt.show()

# External binary eval with the transfer model
y_ext_prob_tl = tl_model.predict(X_ext_m, batch_size=32, verbose=0)
p_stroke_tl = 1.0 - y_ext_prob_tl[:, normal_idx]
y_ext_pred_tl = (p_stroke_tl >= 0.5).astype(int)
print(f"\n[{BACKBONE}] External binary accuracy:", float((y_ext_pred_tl == y_ext_true).mean()))
print(classification_report(y_ext_true, y_ext_pred_tl, target_names=["Normal", "Stroke"], digits=4))


In [ ]:
tl_model.save("stroke_transfer_Efficient_model.h5")

## 10. ConvNeXt-Tiny

ConvNeXt is a modernized pure-CNN (2022) that matches transformer accuracy while keeping CNN efficiency. `ConvNeXtTiny` is ~28M params — comparable to ResNet50 but usually stronger on medical images. Same two-phase recipe: freeze backbone, train head, then unfreeze the top ~30% and fine-tune at a small LR.

Note: ConvNeXt's `preprocess_input` is a no-op — the model does rescaling + normalization internally, so we feed it our `[0, 1]` tensors scaled to `[0, 255]`.


In [ ]:
from tensorflow.keras.applications import ConvNeXtTiny

inputs_cn = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="ct_windows")
x_cn = data_augmentation(inputs_cn)
x_cn = layers.Rescaling(255.0)(x_cn)  # ConvNeXt expects [0,255]; normalization is baked into the model

base_cn = ConvNeXtTiny(include_top=False, weights="imagenet", input_tensor=x_cn)
base_cn.trainable = False  # phase 1: freeze

h_cn = layers.GlobalAveragePooling2D()(base_cn.output)
h_cn = layers.LayerNormalization()(h_cn)
h_cn = layers.Dropout(0.3)(h_cn)
h_cn = layers.Dense(128, activation="gelu")(h_cn)
h_cn = layers.Dropout(0.3)(h_cn)
outputs_cn = layers.Dense(NUM_CLASSES, activation="softmax")(h_cn)

cn_model = models.Model(inputs_cn, outputs_cn, name="convnext_tiny_stroke")
cn_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print(f"ConvNeXt-Tiny | trainable head params: {sum([np.prod(v.shape) for v in cn_model.trainable_weights]):,}")
cn_model.summary()


In [ ]:
pip install tensorflow

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# Phase 1: head training on skull-stripped tensors
cn_cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        filepath="checkpoints/convnext_tiny_head.keras",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

history_cn_head = cn_model.fit(
    X_train_m, y_train,
    validation_data=(X_val_m, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=cn_cbs,
    verbose=1,
)


In [ ]:
# Phase 2: fine-tune the top ~30% of ConvNeXt-Tiny at a small LR
base_cn.trainable = True
n_cn = len(base_cn.layers)
freeze_until_cn = int(n_cn * 0.7)
for lyr in base_cn.layers[:freeze_until_cn]:
    lyr.trainable = False
print(f"Unfrozen layers: {n_cn - freeze_until_cn}/{n_cn}")

cn_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cn_cbs_ft = [
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        filepath="checkpoints/convnext_tiny_finetuned.keras",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]

history_cn_ft = cn_model.fit(
    X_train_m, y_train,
    validation_data=(X_val_m, y_val),
    epochs=10,
    batch_size=16,
    class_weight=class_weights,
    callbacks=cn_cbs_ft,
    verbose=1,
)


In [ ]:
# Evaluate ConvNeXt-Tiny on masked internal test set + external binary eval
cn_loss, cn_acc = cn_model.evaluate(X_test_m, y_test, verbose=0)
print(f"[ConvNeXt-Tiny] Test loss: {cn_loss:.4f}  |  Test accuracy: {cn_acc:.4f}")

y_prob_cn = cn_model.predict(X_test_m, batch_size=32, verbose=0)
y_pred_cn = np.argmax(y_prob_cn, axis=1)

print("\nClassification report:\n")
print(classification_report(y_test_int, y_pred_cn, target_names=target_names, digits=4))

cm_cn = confusion_matrix(y_test_int, y_pred_cn)
cm_cn_norm = cm_cn.astype(np.float32) / cm_cn.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(cm_cn, display_labels=target_names).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("ConvNeXt-Tiny — Confusion (counts)")
sns.heatmap(cm_cn_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names, ax=axes[1], cbar=False)
axes[1].set_title("ConvNeXt-Tiny — Confusion (normalized)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
plt.tight_layout(); plt.show()

# External binary eval (collapse 3-class softmax to Stroke vs Normal)
y_ext_prob_cn = cn_model.predict(X_ext_m, batch_size=32, verbose=0)
p_stroke_cn = 1.0 - y_ext_prob_cn[:, normal_idx]
y_ext_pred_cn = (p_stroke_cn >= 0.5).astype(int)
print(f"\n[ConvNeXt-Tiny] External binary accuracy:", float((y_ext_pred_cn == y_ext_true).mean()))
print(classification_report(y_ext_true, y_ext_pred_cn, target_names=["Normal", "Stroke"], digits=4))


### Summary & Next Steps

**What this notebook does differently from the PNG pipeline**
- Reads raw DICOMs with `pydicom` and recovers **Hounsfield Units** using the rescale tags.
- Builds a **3-channel windowed input** (brain / stroke / bone) instead of an 8-bit grayscale PNG — more physical signal, compatible with ImageNet backbones.
- Validates on a real **held-out external test set** (`External_Test/`) in addition to the internal split.

**Possible improvements**
- Swap the custom CNN for a pretrained **EfficientNetB0 / ResNet50** backbone (the 3-channel input already matches).
- Add **skull stripping** or intracranial-ROI cropping to reduce background variance.
- Use **patient-level splits** (if patient IDs are recoverable from DICOM `PatientID`) to eliminate slice-level leakage.
- Try **focal loss** for the imbalanced multi-class setup.
- For the external binary eval, sweep the decision threshold and report **AUROC / AUPRC** instead of a fixed 0.5 cutoff.

In [ ]:
import numpy as np
import pydicom
from PIL import Image
from scipy import ndimage as ndi

IMG_SIZE = 224
WINDOWS = [(40, 80), (40, 380), (600, 2800)]   # brain, stroke/subdural, bone
MASK_THRESH = 0.02
STRUCT = np.ones((3, 3), dtype=bool)

def skull_strip_dicom(dcm_path, erode_iters=2):
    """Read a DICOM, build the 3-window tensor, apply intracranial mask.
    Returns (masked_tensor[H,W,3] in [0,1], mask[H,W] uint8)."""
    d = pydicom.dcmread(dcm_path)
    arr = d.pixel_array.astype(np.float32)
    slope = float(getattr(d, "RescaleSlope", 1.0))
    intercept = float(getattr(d, "RescaleIntercept", 0.0))
    hu = arr * slope + intercept

    # 3-channel windowed + resized tensor in [0,1]
    chs = []
    for c, w in WINDOWS:
        lo, hi = c - w / 2, c + w / 2
        ch = np.clip((hu - lo) / (hi - lo), 0.0, 1.0)
        ch = np.asarray(
            Image.fromarray((ch * 255).astype(np.uint8)).resize(
                (IMG_SIZE, IMG_SIZE), Image.BILINEAR
            ),
            dtype=np.float32,
        ) / 255.0
        chs.append(ch)
    img = np.stack(chs, axis=-1)  # (H, W, 3)

    # Brain mask from the brain-window channel
    m = img[..., 0] > MASK_THRESH
    m = ndi.binary_fill_holes(m)
    lab, n = ndi.label(m, structure=STRUCT)
    if n > 0:
        sizes = ndi.sum(m, lab, index=np.arange(1, n + 1))
        m = lab == (np.argmax(sizes) + 1)
        m = ndi.binary_fill_holes(m)
        if erode_iters > 0:
            m = ndi.binary_erosion(m, iterations=erode_iters)
    mask = m.astype(np.float32)

    masked = img * mask[..., None]
    return masked, mask.astype(np.uint8)


# --- usage ---
import matplotlib.pyplot as plt

dcm_path = "Brain_Stroke_CT_Dataset/Bleeding/DICOM/10250.dcm"
masked, mask = skull_strip_dicom(dcm_path)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(masked[..., 0] / max(masked[..., 0].max(), 1e-6), cmap="gray")
axes[0].set_title("Original (brain window)"); axes[0].axis("off")
axes[1].imshow(mask, cmap="gray");  axes[1].set_title("Brain mask"); axes[1].axis("off")
axes[2].imshow(masked[..., 0], cmap="gray"); axes[2].set_title("Skull-stripped"); axes[2].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Scan every preprocessed slice, compute its brain-mask area, and pick the
# ones with the LARGEST mask — those slices show the biggest visual difference
# between the raw windowed image and the skull-stripped output.

mask_areas = np.zeros(len(X), dtype=np.int32)
for i in range(len(X)):
    mm = X[i, ..., 0] > MASK_THRESH
    mm = ndi.binary_fill_holes(mm)
    lab, n = ndi.label(mm, structure=STRUCT)
    if n > 0:
        sizes = ndi.sum(mm, lab, index=np.arange(1, n + 1))
        mm = lab == (np.argmax(sizes) + 1)
        mm = ndi.binary_fill_holes(mm)
        mm = ndi.binary_erosion(mm, iterations=2)
    mask_areas[i] = int(mm.sum())

# Top-K slices by absolute mask area
TOP_K = 5
top_idx = np.argsort(mask_areas)[::-1][:TOP_K]

print(f"Total slices scanned: {len(X)}")
print(f"Mask area  — min: {mask_areas.min()}  max: {mask_areas.max()}  "
      f"mean: {mask_areas.mean():.0f}  (out of {IMG_SIZE * IMG_SIZE})")
print(f"\nTop {TOP_K} slices with the largest brain mask:")
for r, i in enumerate(top_idx, 1):
    pct = 100 * mask_areas[i] / (IMG_SIZE * IMG_SIZE)
    print(f"  {r}. idx={i:5d}  area={mask_areas[i]:6d} px ({pct:5.1f}%)  "
          f"label={df.iloc[i]['label']:8s}  path={df.iloc[i]['path']}")

# Visualise the top hit: original (windowed) vs mask vs skull-stripped
fig, axes = plt.subplots(TOP_K, 3, figsize=(11, 3.2 * TOP_K))
if TOP_K == 1:
    axes = axes[None, :]
for r, i in enumerate(top_idx):
    img = X[i]
    mm = img[..., 0] > MASK_THRESH
    mm = ndi.binary_fill_holes(mm)
    lab, n = ndi.label(mm, structure=STRUCT)
    if n > 0:
        sizes = ndi.sum(mm, lab, index=np.arange(1, n + 1))
        mm = lab == (np.argmax(sizes) + 1)
        mm = ndi.binary_fill_holes(mm)
        mm = ndi.binary_erosion(mm, iterations=2)
    mm = mm.astype(np.float32)
    stripped = img * mm[..., None]

    axes[r, 0].imshow(img[..., 0], cmap="gray")
    axes[r, 0].set_title(f"Original  ({df.iloc[i]['label']}, idx={i})", fontsize=9)
    axes[r, 0].axis("off")

    axes[r, 1].imshow(mm, cmap="gray")
    axes[r, 1].set_title(f"Brain mask  ({int(mm.sum())} px)", fontsize=9)
    axes[r, 1].axis("off")

    axes[r, 2].imshow(stripped[..., 0], cmap="gray")
    axes[r, 2].set_title("Skull-stripped", fontsize=9)
    axes[r, 2].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Models
models = [
    "Baseline CNN",
    "EfficientNet-B0",
    "ResNet50",
    "ConvNeXt-Tiny"
]

# Metrics
accuracy = [0.8762, 0.9124, 0.9056, 0.9427]
f1_score = [0.8614, 0.9018, 0.8932, 0.9365]
recall = [0.8421, 0.8950, 0.8874, 0.9284]

# -------------------------
# 1. Accuracy Plot
# -------------------------
plt.figure()
plt.bar(models, accuracy)
plt.title("Model Comparison - Accuracy")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.ylim(0.80, 1.00)
plt.tight_layout()
plt.show()

# -------------------------
# 2. F1-Score Plot
# -------------------------
plt.figure()
plt.bar(models, f1_score)
plt.title("Model Comparison - F1 Score")
plt.ylabel("F1 Score")
plt.xticks(rotation=20)
plt.ylim(0.80, 1.00)
plt.tight_layout()
plt.show()

# -------------------------
# 3. Recall Plot
# -------------------------
plt.figure()
plt.bar(models, recall)
plt.title("Model Comparison - Recall")
plt.ylabel("Recall")
plt.xticks(rotation=20)
plt.ylim(0.80, 1.00)
plt.tight_layout()
plt.show()